# ValveVision — เทรนโมเดลหาจุ๊บวาล์ว (YOLO11n)

**วิธีใช้:** Runtime → Change runtime type → **T4 GPU** แล้วกด Run all

รันทุกเซลล์ตามลำดับ ห้ามข้าม เซลล์สุดท้ายจะพิมพ์รายงานสรุปออกมาเป็นบล็อกเดียว
**ก๊อปบล็อกนั้นทั้งหมดส่งกลับมา** พร้อมโหลดไฟล์ `valve_result.zip`

> ทุกอย่างเก็บลง **Google Drive** ที่ `MyDrive/ValveVision/` ตั้งแต่ระหว่างเทรน
> ถ้า Colab ตัดการเชื่อมต่อกลางคัน ให้รันเซลล์เทรนใหม่ — มันจะเทรนต่อจากจุดเดิมเอง ไม่ต้องเริ่มหนึ่ง

| เซลล์ | ทำอะไร | เวลา |
|---|---|---|
| 1 | ติดตั้ง | ~1 นาที |
| 1.5 | **ต่อ Google Drive — ห้ามข้าม** | ~30 วินาที |
| 2 | โหลด dataset | ~1 นาที |
| 3 | **ตรวจ dataset ก่อนเทรน** (สำคัญ — เช็คว่าแบ่ง train/val รั่วไหม) | ทันที |
| 4 | เทรน | ~40 นาที |
| 5 | หาค่า CONF_THRESH จากชุด valid | ~1 นาที |
| 6 | วัดผลบนชุด test + ภาพลบ | ~2 นาที |
| 7 | export ONNX สำหรับ Pi | ~1 นาที |
| 8 | พิมพ์รายงาน + zip | ทันที |

## 1. ติดตั้งและเช็ค GPU

In [ ]:
!pip install -q ultralytics roboflow onnx onnxslim onnxruntime

import torch, ultralytics, subprocess
print("ultralytics :", ultralytics.__version__)
print("torch       :", torch.__version__)
print("GPU         :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "*** ไม่มี GPU — ไปตั้ง Runtime > Change runtime type > T4 GPU ***")

## 1.5 ต่อ Google Drive — **ห้ามข้าม**

Colab ฟรีตัดการเชื่อมต่อเองได้ทุกเมื่อ แล้ว**ล้างดิสก์ `/content` ทิ้งทั้งหมด** ผลที่พิมพ์บนจอยังอยู่ แต่ไฟล์หายเกลี้ยง

เซลล์นี้ทำให้ทุกอย่างเขียนลง Drive **ตั้งแต่ระหว่างเทรน** ไม่ใช่ตอนจบ ถ้าหลุดที่ epoch 120 น้ำหนักที่ดีที่สุดก็ยังอยู่ใน Drive แล้ว

จะมีหน้าต่างเด้งให้กดอนุญาต — กดยอมรับด้วยบัญชี Google เดียวกับที่เปิด Colab

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

# ทุกผลลัพธ์ลงที่นี่ รอดจากการที่ runtime ถูกล้าง
DRIVE_OUT = "/content/drive/MyDrive/ValveVision"
os.makedirs(DRIVE_OUT, exist_ok=True)
print("ผลลัพธ์จะเก็บที่:", DRIVE_OUT)

# ถ้าเคยเทรนค้างไว้ จะเห็นของเก่าตรงนี้
if os.path.isdir(f"{DRIVE_OUT}/valve/v1/weights"):
    print("\n!! เจอผลการเทรนรอบก่อนใน Drive:")
    for f in sorted(os.listdir(f"{DRIVE_OUT}/valve/v1/weights")):
        print("   ", f)
    print("   ถ้าเทรนใหม่ทับ ให้เปลี่ยน RUN_NAME ในเซลล์เทรนเป็นชื่ออื่น")

## 2. โหลด dataset จาก Roboflow

In [ ]:
from roboflow import Roboflow
import os, glob, yaml

rf      = Roboflow(api_key="Nup7mqlVKUTeENgFgr09")
project = rf.workspace("aooo").project("valve-v1-0-1")
version = project.version(1)
dataset = version.download("yolov11")

DATA_ROOT = dataset.location
print("\nโหลดไว้ที่:", DATA_ROOT)

# --- แก้ data.yaml ให้ชี้ path เต็ม ---
# Roboflow เขียน path แบบสัมพัทธ์ที่ ultralytics บน Colab หาไม่เจอบ่อยๆ
YAML_PATH = os.path.join(DATA_ROOT, "data.yaml")
with open(YAML_PATH) as f:
    dy = yaml.safe_load(f)

for split in ("train", "val", "test"):
    d = os.path.join(DATA_ROOT, "valid" if split == "val" else split, "images")
    if os.path.isdir(d):
        dy[split] = d
    else:
        dy.pop(split, None)
        print(f"!! ไม่มีโฟลเดอร์ {split}")

with open(YAML_PATH, "w") as f:
    yaml.safe_dump(dy, f, sort_keys=False, allow_unicode=True)

print("\ndata.yaml:")
print(yaml.safe_dump(dy, sort_keys=False, allow_unicode=True))

## 2.5 แบ่ง train/valid/test ใหม่ตามรอบ — **อัตโนมัติ ไม่ต้องแตะ Roboflow**

Roboflow สุ่มแบ่งรายภาพ ทำให้ภาพจากรอบเดียวกันไปอยู่ทั้ง train และ test พร้อมกัน — ตัวเลขที่ได้จะสวยเกินจริง

เซลล์นี้เทของทั้ง 3 กองมารวมกัน แล้วแบ่งใหม่**ตามรอบ** (`<นาฬิกา>_<แสง>` ที่อ่านจากชื่อไฟล์) ไม่มีรอบไหนอยู่เกินหนึ่ง split

กันไว้ให้ยากขึ้นด้วย: **2 ตำแหน่งนาฬิกาถูกกันออกจาก train ทั้งหมด** แบ่งไป valid กับ test อย่างละ 1 — โมเดลจึงถูกวัดกับมุมที่ไม่เคยเห็นจริงๆ

> ตั้ง `RESPLIT = False` ถ้าอยากใช้ split ของ Roboflow ตามเดิม

In [ ]:
RESPLIT   = True      # False = ใช้ split เดิมของ Roboflow
VAL_FRAC  = 0.15
TEST_FRAC = 0.15

import os, re, glob, shutil, random, yaml
from collections import defaultdict

if RESPLIT:
    PAT2 = re.compile(r'^(\d+)_(day|room|dim|torch)_(\d+)_(scanA|scanB|nearblur|near|neg)')

    # ── 1. เทมารวมกองเดียว ────────────────────────────────────────────
    pool = []
    for sp in ("train", "valid", "test"):
        for img in glob.glob(os.path.join(DATA_ROOT, sp, "images", "*")):
            lbl = img.replace(f"{os.sep}images{os.sep}", f"{os.sep}labels{os.sep}").rsplit(".", 1)[0] + ".txt"
            m = PAT2.match(os.path.basename(img))
            pool.append({
                "img": img,
                "lbl": lbl if os.path.exists(lbl) else None,
                "clock": m.group(1) if m else "?",
                "light": m.group(2) if m else "?",
                "round": f"{m.group(1)}_{m.group(2)}" if m else "UNKNOWN",
            })

    by_round = defaultdict(list)
    for x in pool:
        by_round[x["round"]].append(x)

    unknown = len(by_round.get("UNKNOWN", []))
    if unknown:
        print(f"!! อ่านชื่อไฟล์ไม่ออก {unknown} ใบ — จะโยนเข้า train ทั้งหมด")

    rounds = sorted(r for r in by_round if r != "UNKNOWN")
    clocks = sorted({r.split("_")[0] for r in rounds}, key=lambda c: int(c))
    print(f"เจอ {len(rounds)} รอบ · {len(clocks)} ตำแหน่งนาฬิกา: {clocks}")

    # ── 2. กัน 2 นาฬิกาออกจาก train ทั้งหมด (เลือกให้ห่างกัน) ─────────
    hold = [clocks[len(clocks)//3], clocks[2*len(clocks)//3]] if len(clocks) >= 3 else clocks[:2]
    val_rounds  = {r for r in rounds if r.split("_")[0] == hold[0]}
    test_rounds = {r for r in rounds if r.split("_")[0] == hold[1]}
    print(f"กันนาฬิกา {hold[0]} ไว้ให้ valid · {hold[1]} ไว้ให้ test (train จะไม่เห็นเลย)")

    # ── 3. เติมรอบอื่นจนได้สัดส่วน โดยสุ่มแบบล็อก seed ให้ทำซ้ำได้ ───
    total = len(pool)
    rest  = [r for r in rounds if r not in val_rounds and r not in test_rounds]
    random.Random(0).shuffle(rest)

    def n_img(rs):
        return sum(len(by_round[r]) for r in rs)

    for r in rest:
        if n_img(val_rounds) < VAL_FRAC * total:
            val_rounds.add(r)
        elif n_img(test_rounds) < TEST_FRAC * total:
            test_rounds.add(r)

    assign = {}
    for r in rounds:
        assign[r] = "valid" if r in val_rounds else ("test" if r in test_rounds else "train")
    assign["UNKNOWN"] = "train"

    # ── 4. ย้ายไฟล์จริง ───────────────────────────────────────────────
    # วางข้างๆ โฟลเดอร์ที่ Roboflow โหลดมา
    NEW = os.path.join(os.path.dirname(DATA_ROOT.rstrip("/")), "dataset_byround")
    shutil.rmtree(NEW, ignore_errors=True)
    for sp in ("train", "valid", "test"):
        os.makedirs(f"{NEW}/{sp}/images", exist_ok=True)
        os.makedirs(f"{NEW}/{sp}/labels", exist_ok=True)

    for x in pool:
        sp = assign[x["round"]]
        shutil.copy(x["img"], f"{NEW}/{sp}/images/{os.path.basename(x['img'])}")
        if x["lbl"]:
            shutil.copy(x["lbl"], f"{NEW}/{sp}/labels/{os.path.basename(x['lbl'])}")

    # ── 5. เขียน data.yaml ใหม่ ────────────────────────────────────────
    old = yaml.safe_load(open(YAML_PATH))
    DATA_ROOT = NEW
    YAML_PATH = f"{NEW}/data.yaml"
    yaml.safe_dump({
        "train": f"{NEW}/train/images",
        "val":   f"{NEW}/valid/images",
        "test":  f"{NEW}/test/images",
        "nc":    old.get("nc", 1),
        "names": old.get("names", ["valve"]),
    }, open(YAML_PATH, "w"), sort_keys=False, allow_unicode=True)

    print()
    print(f"{'split':<8}{'ภาพ':>7}{'%':>8}{'รอบ':>6}   นาฬิกาที่มี")
    for sp in ("train", "valid", "test"):
        rs = [r for r in rounds if assign[r] == sp]
        n  = n_img(rs) + (unknown if sp == "train" else 0)
        cl = sorted({r.split('_')[0] for r in rs}, key=lambda c: int(c))
        print(f"{sp:<8}{n:>7}{100*n/total:>7.1f}%{len(rs):>6}   {','.join(cl)}")
    print(f"\nแบ่งใหม่แล้ว -> {YAML_PATH}")
    print("รันเซลล์ 3 (ตรวจ dataset) ต่อได้เลย ควรขึ้น 'OK ไม่มีรอบรั่วข้าม split'")
else:
    print("ข้าม — ใช้ split เดิมของ Roboflow")

## 3. ตรวจ dataset ก่อนเทรน

เซลล์นี้ตอบ 3 คำถามที่ถ้าผิดแล้วเทรนไปก็เสียเวลาเปล่า:

1. **รอบเดียวกันไปโผล่หลาย split ไหม** — ชื่อไฟล์คือ `<นาฬิกา>_<แสง>_<ลำดับ>_<ชนิด>` เช่น `6_room_012_near`
   "รอบ" = `<นาฬิกา>_<แสง>` ถ้ารอบเดียวกันกระจายไปทั้ง train และ test → ตัวเลขที่ได้จะสวยเกินจริง
2. **ภาพลบเหลืออยู่กี่ใบ** — ถ้าเป็น 0 แปลว่า Roboflow ทิ้งไปตอน generate
3. **สัดส่วนภาพลบเกิน 15% ไหม** — เกินแล้ว recall จะตก

In [ ]:
import os, re, glob
from collections import Counter, defaultdict

SPLITS = ["train", "valid", "test"]
PAT = re.compile(r'^(\d+)_(day|room|dim|torch)_(\d+)_(scanA|scanB|nearblur|near|neg)')

# kind ที่ถือว่าเป็น "ระยะใกล้ 10-20 ซม." — ตรงกับ NEAR_DISTANCES ใน collect_dataset.py
NEAR_KINDS = {"near", "nearblur"}

inv = {}   # split -> list of dict(stem, round, kind, n_box)

def label_path(img_path):
    return img_path.replace(os.sep + "images" + os.sep, os.sep + "labels" + os.sep).rsplit(".", 1)[0] + ".txt"

for sp in SPLITS:
    d = os.path.join(DATA_ROOT, sp, "images")
    rows = []
    for p in sorted(glob.glob(os.path.join(d, "*"))):
        stem = os.path.basename(p)
        m = PAT.match(stem)
        lp = label_path(p)
        nb = 0
        if os.path.exists(lp):
            nb = sum(1 for line in open(lp) if line.strip())
        rows.append({
            "path":  p,
            "stem":  stem,
            "round": f"{m.group(1)}_{m.group(2)}" if m else "UNKNOWN",
            "kind":  m.group(4) if m else "UNKNOWN",
            "nbox":  nb,
        })
    inv[sp] = rows

AUDIT = []
def say(s=""):
    AUDIT.append(s); print(s)

say("=" * 62)
say("ตรวจ DATASET")
say("=" * 62)

tot = sum(len(v) for v in inv.values())
say(f"ภาพทั้งหมด {tot} ใบ")
say("")
say(f"{'split':<8}{'ภาพ':>7}{'%':>7}{'มีกรอบ':>9}{'ภาพลบ':>8}{'%ลบ':>7}{'รอบ':>6}")
for sp in SPLITS:
    r = inv[sp]
    if not r:
        say(f"{sp:<8}{'— ไม่มี —':>7}"); continue
    pos = sum(1 for x in r if x["nbox"] > 0)
    neg = len(r) - pos
    say(f"{sp:<8}{len(r):>7}{100*len(r)/tot:>6.1f}%{pos:>9}{neg:>8}{100*neg/len(r):>6.1f}%"
        f"{len(set(x['round'] for x in r)):>6}")

# --- 1. เช็ครอบรั่วข้าม split ---
round2splits = defaultdict(set)
for sp in SPLITS:
    for x in inv[sp]:
        round2splits[x["round"]].add(sp)
leaked = {r: s for r, s in round2splits.items() if len(s) > 1}

say("")
say("-" * 62)
if leaked:
    say(f"XX รอบรั่วข้าม split: {len(leaked)} จาก {len(round2splits)} รอบ")
    for r, s in sorted(leaked.items())[:12]:
        say(f"     {r:<12} อยู่ใน {sorted(s)}")
    if len(leaked) > 12:
        say(f"     ... อีก {len(leaked)-12} รอบ")
    say("")
    say("   >> Roboflow สุ่มแบ่งรายภาพ ซึ่งผิดกฎใน PLAN.md:216")
    say("      ภาพจากรอบเดียวกันเกือบเหมือนกัน พอไปอยู่ทั้ง train และ test")
    say("      ตัวเลข mAP จะสวยเกินจริง ต้องกลับไปแบ่งใหม่ตามรอบ")
    say("      (เทรนต่อได้ แต่ห้ามเอาเลขนี้ไปใส่รายงาน)")
else:
    say("OK ไม่มีรอบรั่วข้าม split")

# --- 2 & 3. ภาพลบ ---
say("-" * 62)
neg_all = sum(1 for sp in SPLITS for x in inv[sp] if x["nbox"] == 0)
if neg_all < 20:
    say(f"XX ภาพลบมีแค่ {neg_all} ใบ — Roboflow ตัดภาพที่ไม่มี label ทิ้งตอน generate")
    say("   >> เกณฑ์ false positive <= 5% จะวัดไม่ได้เลย")
    say("   >> แก้: ตอนอัปโหลดต้องกดยืนยันภาพที่ไม่มีกรอบเป็น background")
else:
    pct = 100 * neg_all / tot
    flag = "OK" if pct <= 15 else "XX"
    say(f"{flag} ภาพลบ {neg_all} ใบ = {pct:.1f}% ของทั้งชุด (ควรอยู่ราว 10-15%)")

# --- ส่วนประกอบตามชนิดภาพ ---
say("-" * 62)
say("ชนิดภาพในแต่ละ split (scanA/scanB = ท่าสแกน · near/nearblur = ระยะใกล้ · neg = ภาพลบ)")
kinds = sorted({x["kind"] for sp in SPLITS for x in inv[sp]})
say(f"{'split':<8}" + "".join(f"{k:>11}" for k in kinds))
for sp in SPLITS:
    c = Counter(x["kind"] for x in inv[sp])
    say(f"{sp:<8}" + "".join(f"{c.get(k,0):>11}" for k in kinds))

unk = sum(1 for sp in SPLITS for x in inv[sp] if x["kind"] == "UNKNOWN")
if unk:
    say(f"\n!! อ่านชื่อไฟล์ไม่ออก {unk} ใบ — ตัวเลขแยกตามระยะจะไม่ครบ")
say("=" * 62)

## 4. เทรน

พารามิเตอร์ตาม `PLAN.md` หัวข้อ D3:

| ค่า | ตั้งไว้ | ทำไม |
|---|---|---|
| `imgsz=640` | ต้องตรงกับ `INPUT_SIZE` ใน `valve_detector.py` | |
| `degrees=5` | J5 ถูกตรึงที่ 90° กล้องไม่หมุนรอบแกนตัวเอง ภาพเอียงมากไม่เกิดจริง | |
| `close_mosaic=15` | ปิด mosaic 15 epoch สุดท้าย ให้เห็นภาพจริงล้วนก่อนจบ | |
| `patience=50` | ถ้า 50 epoch แล้วไม่ดีขึ้น หยุดเอง | |

ถ้าเซลล์ 3 ขึ้น `XX รอบรั่ว` ยังรันต่อได้ (ไว้ดูว่าโมเดลเรียนได้ไหม) แต่ตัวเลขห้ามเอาไปใส่รายงาน

In [ ]:
from ultralytics import YOLO
import os, torch, time

EPOCHS   = 150
RUN_NAME = "v2_byround"   # ★ เปลี่ยนชื่อทุกครั้งที่ dataset เปลี่ยน ไม่งั้นทับของเก่า
RESUME   = False          # ★ ตั้ง True เฉพาะตอนที่ "โดน Colab ตัดกลางคัน" เท่านั้น

LAST = f"{DRIVE_OUT}/valve/{RUN_NAME}/weights/last.pt"

# ── ตัดสินใจว่าจะเทรนต่อได้จริงไหม ──────────────────────────────────────
# ★ บทเรียน 2026-08-28: เดิมเขียนว่า "ถ้ามี last.pt ให้ resume" แล้วมันไปเจอ
#   checkpoint ของรอบที่เทรนจบแล้ว ultralytics resume ไม่ได้จึงตกกลับไปใช้
#   ค่าเริ่มต้นทั้งชุด (data=coco8.yaml, epochs=100) เทรนผิด dataset เงียบๆ
#   ตอนนี้ต้องเช็คว่า checkpoint ยังเทรนไม่จบจริง (epoch >= 0) ก่อนเท่านั้น
can_resume = False
if RESUME and os.path.exists(LAST):
    ck   = torch.load(LAST, map_location="cpu", weights_only=False)
    done = ck.get("epoch", -1)
    if done is not None and done >= 0:
        can_resume = True
        print(f"เจอ checkpoint ที่ค้างอยู่ที่ epoch {done} -> เทรนต่อ")
    else:
        print("checkpoint นี้เทรนจบไปแล้ว resume ไม่ได้ -> เริ่มใหม่")

t0 = time.time()

if can_resume:
    results = YOLO(LAST).train(resume=True)
else:
    results = YOLO("yolo11n.pt").train(
        data=YAML_PATH,                 # <-- ต้องระบุเสมอ ห้ามให้ตกไปใช้ค่าเริ่มต้น
        epochs=EPOCHS,
        imgsz=640,
        batch=16,
        degrees=5,
        close_mosaic=15,
        patience=50,
        seed=0,
        cache=True,
        project=f"{DRIVE_OUT}/valve",
        name=RUN_NAME,
        exist_ok=True,
        plots=True,
    )

TRAIN_MIN = (time.time() - t0) / 60
RUN_DIR   = str(results.save_dir)
BEST      = os.path.join(RUN_DIR, "weights", "best.pt")

# ── ตรวจว่าเทรนถูก dataset จริง ─────────────────────────────────────────
used = str(getattr(results, "save_dir", ""))
names = YOLO(BEST).names
print(f"\nเทรนเสร็จใน {TRAIN_MIN:.1f} นาที -> {BEST}")
print(f"คลาสในโมเดล: {names}")

if len(names) != 1:
    raise RuntimeError(
        f"XX โมเดลมี {len(names)} คลาส แต่โปรเจ็คนี้มีคลาสเดียว\n"
        f"   แปลว่าเทรนผิด dataset (น่าจะตกไปใช้ coco8.yaml)\n"
        f"   ลบโฟลเดอร์ {DRIVE_OUT}/valve/{RUN_NAME} ทิ้ง แล้วรันเซลล์นี้ใหม่")
if TRAIN_MIN < 5:
    raise RuntimeError(
        f"XX เทรนเสร็จใน {TRAIN_MIN:.1f} นาที ซึ่งเร็วเกินจริงสำหรับ ~880 ภาพ\n"
        f"   แปลว่าโหลด dataset ไม่ครบ ตรวจ YAML_PATH = {YAML_PATH}")
print("OK เทรนถูก dataset")


## 5. หาค่า `CONF_THRESH` จากชุด valid

`CONF_THRESH` คือเส้นแบ่งว่าคะแนนความมั่นใจเท่าไรถึงจะนับว่า "เจอจุ๊บ" ตอนนี้ในโค้ดตั้งไว้ 0.10 ซึ่งต่ำมาก

เซลล์นี้หา 2 ค่าแล้วให้เลือก:
- **ค่าที่ F1 สูงสุด** — สมดุลระหว่างเจอครบกับไม่ทายมั่ว
- **ค่าที่ต่ำที่สุดที่ยัง recall ≥ 0.95** — ตรงกับเกณฑ์ที่สำคัญที่สุดของโปรเจ็ค

**ต้องใช้ชุด valid เท่านั้น** ถ้าใช้ test เลือกค่า ตัวเลข test จะไม่ใช่ของจริงอีกต่อไป

In [ ]:
import numpy as np

model = YOLO(BEST)
mv = model.val(data=YAML_PATH, split="val", imgsz=640, plots=True, project=f"{DRIVE_OUT}/valve", name="val_valid", exist_ok=True)

VAL_DIR = str(mv.save_dir)

# ultralytics เก็บเส้นโค้งไว้ที่ mv.curves_results แต่ละตัวคือ [x, y, ชื่อแกน x, ชื่อแกน y]
# มี 4 เส้น: Precision-Recall, F1-Confidence, Precision-Confidence, Recall-Confidence
# ★ ต้องกรองด้วยแกน x = "Confidence" ด้วย ไม่งั้นจะไปหยิบเส้น PR ที่แกน x เป็น Recall มาแทน
def curve(ylabel):
    return next((c for c in mv.curves_results
                 if str(c[2]).startswith("Confidence") and str(c[3]).startswith(ylabel)), None)

f1c = curve("F1")
pc  = curve("Precision")
rc  = curve("Recall")
print("เส้นโค้งที่มี:", [f"{c[2]} -> {c[3]}" for c in mv.curves_results])

conf_f1 = conf_r95 = None
REPORT_VAL = []
def sayv(s=""):
    REPORT_VAL.append(s); print(s)

sayv("=" * 62)
sayv("ชุด VALID — ใช้เลือกค่า CONF_THRESH")
sayv("=" * 62)
sayv(f"mAP50    {mv.box.map50:.4f}")
sayv(f"mAP50-95 {mv.box.map:.4f}")

if f1c is not None:
    x  = np.array(f1c[0], dtype=float)
    f1 = np.array(f1c[1], dtype=float).ravel()[:len(x)]
    conf_f1 = float(x[int(np.argmax(f1))])
    sayv(f"\nF1 สูงสุด {f1.max():.4f} ที่ conf = {conf_f1:.3f}")

if rc is not None:
    x = np.array(rc[0], dtype=float)
    r = np.array(rc[1], dtype=float).ravel()[:len(x)]
    ok = np.where(r >= 0.95)[0]
    if len(ok):
        conf_r95 = float(x[ok[-1]])       # conf สูงสุดที่ recall ยัง >= 0.95
        p_here = float(np.array(pc[1], dtype=float).ravel()[ok[-1]]) if pc is not None else float("nan")
        sayv(f"recall >= 0.95 ได้ถึง conf = {conf_r95:.3f} (precision ตรงนั้น {p_here:.3f})")
    else:
        sayv(f"!! recall ไม่ถึง 0.95 ที่ conf ใดเลย (สูงสุด {r.max():.4f}) — dataset ยังไม่พอ")

    sayv("\nตาราง conf -> precision / recall / F1")
    sayv(f"{'conf':>7}{'precision':>12}{'recall':>10}{'F1':>9}")
    for t in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60, 0.70]:
        i = int(np.argmin(np.abs(x - t)))
        pp = float(np.array(pc[1], dtype=float).ravel()[i]) if pc is not None else float("nan")
        ff = float(np.array(f1c[1], dtype=float).ravel()[i]) if f1c is not None else float("nan")
        sayv(f"{t:>7.2f}{pp:>12.4f}{r[i]:>10.4f}{ff:>9.4f}")

# เลือกจุดทำงาน: ดันให้ recall สูงที่สุด "โดยที่ precision ยังไม่ต่ำกว่าพื้น"
# เหตุผลที่ต้องมีพื้น: recall จะสูงสุดเสมอที่ conf ~= 0 แต่ตรงนั้น precision พังไปด้วย
# ไม่ใช่จุดทำงานจริง — ยอม precision ต่ำได้ระดับหนึ่งเพราะ PLAN.md:210 มีตัวกรองซ้ำ
# อยู่แล้ว (ผลตรวจจับที่ไม่อยู่บนรัศมีวงจุ๊บถูกตัดทิ้งฟรี)
P_FLOOR = 0.80

CONF = None
if pc is not None and rc is not None:
    x  = np.array(rc[0], dtype=float)
    r  = np.array(rc[1], dtype=float).ravel()[:len(x)]
    pp = np.array(pc[1], dtype=float).ravel()[:len(x)]
    ok = np.where(pp >= P_FLOOR)[0]
    if len(ok):
        i    = int(ok[0])                 # conf ต่ำสุดที่ precision ยังถึงพื้น = recall สูงสุด
        CONF = float(x[i])
        sayv(f"\n>> เลือก conf = {CONF:.3f}  (precision {pp[i]:.4f} · recall {r[i]:.4f})")
        sayv(f"   กฎ: recall สูงสุดโดยที่ precision >= {P_FLOOR}")
        if r[i] < 0.95:
            sayv(f"   !! recall {r[i]:.4f} ยังไม่ถึงเกณฑ์ 0.95 ของ PLAN.md")
            sayv( "      โมเดลยังไม่ผ่าน ไม่ใช่เพราะตั้ง conf ผิด — ต้องแก้ที่ dataset")
    else:
        sayv(f"\n!! precision ไม่ถึง {P_FLOOR} ที่ conf ใดเลย — โมเดลยังใช้ไม่ได้")

if CONF is None:
    CONF = conf_f1 if conf_f1 is not None else 0.25
    sayv(f"\n>> ใช้ค่าที่ F1 สูงสุดแทน: conf = {CONF:.3f}")

CONF = round(float(CONF), 3)
sayv(f"\n>> CONF_THRESH = {CONF}  (ใช้วัดผลเซลล์ถัดไป)")
sayv("=" * 62)


## 6. วัดผลบนชุด test + ภาพลบ

ตัวเลขในเซลล์นี้คือตัวเลขที่เอาไปใส่รายงานได้ — เพราะเลือก `CONF_THRESH` จบไปแล้วจากชุด valid

วัด 3 อย่างตามเกณฑ์ผ่านใน `PLAN.md`:

| ตัวชี้วัด | เกณฑ์ |
|---|---|
| mAP50 บนชุด test | >= 0.85 |
| recall ที่ระยะใกล้ (ภาพ `near` + `nearblur` = 10–20 ซม.) | **>= 0.95** |
| false positive บนภาพลบ | <= 5% |

> ระยะไม่ได้ถูกเก็บไว้ในชื่อไฟล์ แต่ `NEAR_DISTANCES` ใน `collect_dataset.py` คือ
> `[100, 120, 140, 170, 200]` มม. **ภาพ `near` ทุกใบจึงอยู่ในช่วง 10–20 ซม. อยู่แล้ว**
> recall ของภาพ `near` จึงใช้แทนเกณฑ์นี้ได้ตรงๆ

In [ ]:
import numpy as np
from collections import defaultdict
from PIL import Image as PImage

EVAL_SPLIT = "test" if inv.get("test") else "valid"
model = YOLO(BEST)

mt = model.val(data=YAML_PATH, split=("test" if EVAL_SPLIT == "test" else "val"),
               imgsz=640, plots=True, project=f"{DRIVE_OUT}/valve", name="val_test", exist_ok=True)
TEST_DIR = str(mt.save_dir)

# ---------- ตัวจับคู่กรอบเอง เพื่อแยก recall ตามชนิดภาพ ----------
def xywhn_to_xyxy(b, w, h):
    cx, cy, bw, bh = b
    return [(cx - bw/2)*w, (cy - bh/2)*h, (cx + bw/2)*w, (cy + bh/2)*h]

def iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, x2-x1) * max(0.0, y2-y1)
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter/ua if ua > 0 else 0.0

IOU_MATCH = 0.5
rows = inv[EVAL_SPLIT]
paths = [x["path"] for x in rows]

preds = {}
B = 32
for i in range(0, len(paths), B):
    chunk = paths[i:i+B]
    for p, res in zip(chunk, model.predict(chunk, imgsz=640, conf=CONF, verbose=False)):
        preds[p] = res.boxes.xyxy.cpu().numpy().tolist() if res.boxes is not None else []

stat = defaultdict(lambda: {"gt": 0, "hit": 0, "img": 0, "img_hit": 0, "fp_img": 0})
for x in rows:
    k = x["kind"]
    s = stat[k]
    s["img"] += 1
    lp = label_path(x["path"])
    gts = []
    if os.path.exists(lp):
        W, H = PImage.open(x["path"]).size
        for line in open(lp):
            f = line.split()
            if len(f) >= 5:
                gts.append(xywhn_to_xyxy([float(v) for v in f[1:5]], W, H))
    pr = preds.get(x["path"], [])
    if not gts:
        if pr:
            s["fp_img"] += 1
        continue
    s["gt"] += len(gts)
    used = set()
    for g in gts:
        best, bi = 0.0, -1
        for j, b in enumerate(pr):
            if j in used:
                continue
            v = iou(g, b)
            if v > best:
                best, bi = v, j
        if best >= IOU_MATCH:
            used.add(bi); s["hit"] += 1
    if used:
        s["img_hit"] += 1

REPORT_TEST = []
def sayt(s=""):
    REPORT_TEST.append(s); print(s)

sayt("=" * 62)
sayt(f"ชุด {EVAL_SPLIT.upper()} — ตัวเลขสำหรับรายงาน (CONF_THRESH = {CONF})")
sayt("=" * 62)
sayt(f"mAP50     {mt.box.map50:.4f}   เกณฑ์ >= 0.85   {'ผ่าน' if mt.box.map50 >= 0.85 else 'ไม่ผ่าน'}")
sayt(f"mAP50-95  {mt.box.map:.4f}")
sayt(f"precision {float(mt.box.mp):.4f}")
sayt(f"recall    {float(mt.box.mr):.4f}")

sayt("")
sayt("แยกตามชนิดภาพ")
sayt(f"{'ชนิด':<12}{'ภาพ':>6}{'กรอบจริง':>10}{'เจอ':>7}{'recall':>10}")
for k in sorted(stat):
    s = stat[k]
    if s["gt"] == 0:
        continue
    sayt(f"{k:<12}{s['img']:>6}{s['gt']:>10}{s['hit']:>7}{s['hit']/s['gt']:>10.4f}")

near_gt  = sum(stat[k]["gt"]  for k in NEAR_KINDS if k in stat)
near_hit = sum(stat[k]["hit"] for k in NEAR_KINDS if k in stat)
scan_gt  = sum(stat[k]["gt"]  for k in ("scanA", "scanB") if k in stat)
scan_hit = sum(stat[k]["hit"] for k in ("scanA", "scanB") if k in stat)

sayt("")
if near_gt:
    r = near_hit / near_gt
    sayt(f">> recall ระยะใกล้ 10-20 ซม. = {r:.4f}  ({near_hit}/{near_gt})   เกณฑ์ >= 0.95   {'ผ่าน' if r >= 0.95 else 'ไม่ผ่าน'}")
else:
    sayt("!! ไม่มีภาพระยะใกล้ในชุดนี้")
if scan_gt:
    sayt(f"   recall ท่าสแกน        = {scan_hit/scan_gt:.4f}  ({scan_hit}/{scan_gt})")

# ---------- false positive บนภาพลบ (ทุก split รวมกัน เพื่อให้ตัวเลขนิ่ง) ----------
neg_paths = [x["path"] for sp in SPLITS for x in inv[sp] if x["nbox"] == 0]
FP_RATE = None
if neg_paths:
    fp = 0
    for i in range(0, len(neg_paths), B):
        for res in model.predict(neg_paths[i:i+B], imgsz=640, conf=CONF, verbose=False):
            if res.boxes is not None and len(res.boxes):
                fp += 1
    FP_RATE = fp / len(neg_paths)
    sayt("")
    sayt(f">> false positive บนภาพลบ = {FP_RATE:.4f} ({fp}/{len(neg_paths)} ใบ)   เกณฑ์ <= 0.05   {'ผ่าน' if FP_RATE <= 0.05 else 'ไม่ผ่าน'}")
else:
    sayt("\n!! ไม่มีภาพลบให้วัด")
sayt("=" * 62)

## 7. export ONNX สำหรับ Pi

In [ ]:
onnx_path = YOLO(BEST).export(format="onnx", imgsz=640, simplify=True, opset=12)
print("\nได้ไฟล์:", onnx_path)

# เช็คว่า onnxruntime บน Pi จะโหลดได้จริงและรูปร่าง input/output ตรงกับ valve_detector.py
import onnxruntime as ort, numpy as np
sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
i = sess.get_inputs()[0]; o = sess.get_outputs()[0]
ONNX_IO = f"input {i.name} {i.shape} · output {o.name} {o.shape}"
print(ONNX_IO)
out = sess.run(None, {i.name: np.zeros((1, 3, 640, 640), dtype=np.float32)})[0]
print("ลองรันเปล่า ผ่าน · output shape =", out.shape)

# คลาสเดียวต้องได้ 5 ช่อง (4 พิกัด + 1 คะแนน) ถ้าได้ 84 คือโมเดล COCO 80 คลาส
if out.shape[1] != 5:
    raise RuntimeError(f"XX output มี {out.shape[1]} ช่อง ควรเป็น 5 — เทรนผิด dataset")
print("OK output 5 ช่อง = คลาสเดียวถูกต้อง")

## 8. รายงานสรุป + ดาวน์โหลด

**ก๊อปข้อความทั้งบล็อกที่พิมพ์ออกมาส่งกลับมาให้ผม** แล้วโหลด `valve_result.zip` เก็บไว้

ในไฟล์ zip มี: `best.pt` · `best.onnx` · `results.csv` · กราฟทั้งหมด (PR curve, F1 curve, confusion matrix, ตัวอย่างผลทำนาย) · `args.yaml`

In [ ]:
import shutil, os, glob, json

STAGE = "/content/valve_result"
shutil.rmtree(STAGE, ignore_errors=True)
os.makedirs(STAGE, exist_ok=True)

for src, dst in [(BEST, "best.pt"), (str(onnx_path), "best.onnx")]:
    if os.path.exists(src):
        shutil.copy(src, os.path.join(STAGE, dst))

for d, tag in [(RUN_DIR, "train"), (VAL_DIR, "val_valid"), (TEST_DIR, "val_test")]:
    for p in glob.glob(os.path.join(d, "*.png")) + glob.glob(os.path.join(d, "*.jpg")) \
           + glob.glob(os.path.join(d, "*.csv")) + glob.glob(os.path.join(d, "*.yaml")):
        shutil.copy(p, os.path.join(STAGE, f"{tag}__{os.path.basename(p)}"))

LINES = []
LINES.append("#" * 62)
LINES.append("#  ValveVision — ผลการเทรน  (ก๊อปทั้งบล็อกนี้ส่งกลับ)")
LINES.append("#" * 62)
LINES.append("")
LINES.append(f"ultralytics {ultralytics.__version__} · {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
LINES.append(f"dataset: aooo/valve-v1-0-1 v1 · เทรน {EPOCHS} epochs · ใช้เวลา {TRAIN_MIN:.1f} นาที")
LINES.append(f"เกณฑ์ที่เลือกใช้: CONF_THRESH = {CONF}")
LINES.append(f"ONNX: {ONNX_IO}")
LINES.append("")
LINES += AUDIT
LINES.append("")
LINES += REPORT_VAL
LINES.append("")
LINES += REPORT_TEST
LINES.append("")

# --- ผลรายและ epoch ท้ายๆ เพื่อดูว่ายังไม่อิ่มตัวหรือ overfit ---
csv_p = os.path.join(RUN_DIR, "results.csv")
if os.path.exists(csv_p):
    import pandas as pd
    df = pd.read_csv(csv_p)
    df.columns = [c.strip() for c in df.columns]
    keep = [c for c in ["epoch", "train/box_loss", "val/box_loss",
                        "metrics/precision(B)", "metrics/recall(B)",
                        "metrics/mAP50(B)", "metrics/mAP50-95(B)"] if c in df.columns]
    LINES.append("=" * 62)
    LINES.append("เส้นทางการเทรน (10 epoch สุดท้าย) — ดูว่า val loss เริ่มขึ้นหรือยัง")
    LINES.append("=" * 62)
    LINES.append(df[keep].tail(10).to_string(index=False))
    best_i = df["metrics/mAP50(B)"].idxmax() if "metrics/mAP50(B)" in df else None
    if best_i is not None:
        LINES.append(f"\nmAP50 ดีที่สุดที่ epoch {int(df.loc[best_i,'epoch'])} = {df.loc[best_i,'metrics/mAP50(B)']:.4f}")
    LINES.append("")

LINES.append("#" * 62)
LINES.append("#  จบรายงาน")
LINES.append("#" * 62)

REPORT = "\n".join(str(x) for x in LINES)
with open(os.path.join(STAGE, "report.txt"), "w") as f:
    f.write(REPORT)

shutil.make_archive("/content/valve_result", "zip", STAGE)
shutil.copy("/content/valve_result.zip", f"{DRIVE_OUT}/valve_result.zip")
print(f">> เก็บสำเนาไว้ที่ {DRIVE_OUT}/valve_result.zip แล้ว (ไม่หายแม้ runtime ถูกล้าง)")
print(REPORT)
print("\n\nไฟล์ใน zip:")
for p in sorted(os.listdir(STAGE)):
    print("  ", p)

from google.colab import files
try:
    files.download("/content/valve_result.zip")
except Exception as e:
    print("ดาวน์โหลดอัตโนมัติไม่ผ่าน:", e)
    print("ไม่เป็นไร ไปโหลดเองจาก Google Drive > ValveVision > valve_result.zip")